## Generate Pyrosetta Stubs

### Verify pyrosetta install (and that vscode/pylance/etc doesn't autocomplete imports and properties)

In [ ]:
import pyrosetta; pyrosetta.init()
from pyrosetta.rosetta.core.pose import Pose
p = Pose()
p?

### Install pybind11-stubgen (the star of the show)
pybind11-stubgen is a python script which imports a module, analyzes its contents at runtime, uses annotations and docstring information to infer types, and produces a tree of "type stub files" (*.pyi extension, see [the documentation](https://typing.python.org/en/latest/spec/distributing.html#packaging-typed-libraries)). For a library like pyrosetta, which exposes modules to the Python interpreter not through files but through a C++ interface, this creates a static description in Python of the contents of pyrosetta which is parseable by IDEs and static type checkers like VSCode and Pylance.

In [ ]:
%pip install pybind11-stubgen

### Run pybind11-stubgen on pyrosetta into current directory
This should let VSCode see pyrosetta types in the current workspace. Try it out by typing `from pyrosetta.rosetta.core.pose import Pose` using VSCode's autocomplete! You may need to close/reopen VSCode for the changes to take effect.

In [ ]:
%run -m pybind11-stubgen pyrosetta -o .

You will likely see a very large number of error messages, followed by `pybind11_stubgen - [WARNING] Raw C++ types/values were found in signatures extracted from docstrings.` This is a consequence of the way pybind11 (used to build pyrosetta) generates the docstrings which are used by pybind11-stubgen to generate type information; unfortunately, during the binding process certain objects will be exported to python before the types of their attributes/parameters/return values are, meaning pybind11 cannot assign them their proper python type in the docstring You can see this if you inspect the objects' docstrings at runtime through, for example `Pose.add_constraint?` [in ipython/jupyter] or `help(Pose.add_constraint)` [in any python]; you'll see the type of the "constraint" parameter (and return value) as the C++ type `core::scoring::constraints::Constraint` as opposed to the python equivalent `pyrosetta.rosetta.core.scoring.constraints.Constraint`. This in turn means pybind11-stubgen cannot infer types for those attributes/parameters/return values. Luckily, names, module contents, and class contents are unaffected, and only a minority of annotations are in this way affected. Additionally, as you can see above, it's usually easy to convert the docstring type into the python type by simply prepending the import chain with `pyrosetta.rosetta`. Theoretically, it would be possible to change the binding process to automatically resolve these binding-order issues, but doing so would require a significant update to [Binder](https://github.com/RosettaCommons/binder), the software used to automate the binding process for rosetta.

TL;DR, the errors are acceptable and expected.

## Install Stubfiles into site-packages
This will let vscode and other static type checkers see the stub files as being part of the pyrosetta install

### Find site-packages folder
You might need a different site_packages folder if you have packages in multiple locations. I *think* vscode can find stubs and source for the same package across different site-packages folders, so if you can't modify the system site-packages (which might happen if, for example, you're using a system-managed python on a cluster), you might be able to install the stubs to a userspace site-packages instead. To do so simply modify the site_packages variable below

In [ ]:
import site
site_packages = site.getsitepackages()[0]
print(site_packages)

### Copy Rosetta stubs into pyrosetta site-packages
Only the pyrosetta/rosetta directory needs stubs, all other files are pure python

In [ ]:
import shutil
import os
dest_path = os.path.join(site_packages,'pyrosetta/rosetta')
shutil.copytree('pyrosetta/rosetta',dest_path,dirs_exist_ok=True)

### [Optional] Cleanup Workspace Stubs
Since local stubs will take priority over site-packages

In [ ]:
shutil.rmtree('pyrosetta')